In [ ]:
!git clone https://github.com/ShathaTm/LK-Hadith-Corpus.git

Cloning into 'LK-Hadith-Corpus'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (388/388), done.
remote: Total 409 (delta 48), reused 351 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 13.69 MiB | 13.37 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [ ]:
import glob

path = '/content/LK-Hadith-Corpus'
files= sorted(glob.glob(path + '//**//*.csv', recursive= True))
print(files)

['/content/LK-Hadith-Corpus/AbuDaud/Chapter1.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter10.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter11.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter12.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter13.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter14.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter15.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter16.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter17.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter18.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter19.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter2.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter20.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter21.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter22.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter23.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter24.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter25.csv', '/content/LK-Hadith-Corpus/AbuDaud/Chapter26.csv', '/content/LK-Hadith-Corpus/AbuDa

In [ ]:
import re

def clean_text(text):
  text= text.lower()
  text= re.sub(r'[^a-zA-Z0-9\]','',text)
  text= re.sub(r'[^a-zA-Z]','',text)
  return text

In [ ]:
columns= [
    'Chapter_Number', 'Chanpter_English', 'Chapter_Arabic',
    'Section_Number', 'Section_English', 'Section_Arabic',
    'Hadith_Number',
    'English_Hadith', 'English_Isnad', "English_Matn",
    'Arabic_Hadith', 'Arabic_Isnad', 'Arabic_Matn',
    'Arabic_Grade', 'English_Grade'
]

columns.append('Cleaned_Hadith')

In [ ]:
import pandas as pd
import re

def clean_text(text):
  text= text.lower()
  text= re.sub(r'[^a-zA-Z]','',text) # Fix: corrected regex to keep only alphabetic characters
  return text

all_hadith = [ ]
for file in files:
  df= pd.read_csv(file, names= columns, skiprows= 1)
  df['Cleaned_Hadith'] = df['English_Hadith'].astype(str).apply(clean_text)
  # all_hadith.append()
  all_hadith.extend(df[columns].values.tolist())

In [ ]:
hadith_df= pd.DataFrame(all_hadith, columns= columns)
#hadith_df.head()

hadith_df.to_csv('cleaned_hadith.csv', index= False)

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
model= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings= model.encode(hadith_df['Cleaned_Hadith'].values)

In [ ]:
import numpy as np
embeddings= np.array(embeddings)


In [ ]:
np.save('hadith_embeddings.npy', embeddings)
embeddings = np.load('hadith_embeddings.npy')

In [ ]:
# !pip install faiss-gpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.9 MB/s eta 0:00:00


In [ ]:
import faiss

dimensions= embeddings.shape[1]
faiss_index= faiss.IndexFlatL2(dimensions)

faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'faiss_index.index')

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def get_similar_hadith(query, count= 5, model = model, faiss_index= faiss_index):
  query_embedding= model.encode([query])
  distance, indices= faiss_index.search(query_embedding, count)


  for i in range(count):
    print(hadith_df['English_Hadith'].iloc[indices[0][i]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
